# Football Object Detection — YOLOv8x Fine-Tuning

**Person A: CV, Object Detection & Camera Mechanics Lead**

This notebook fine-tunes YOLOv8x on a football dataset from Roboflow.

**Classes:**
- 0: ball
- 1: goalkeeper
- 2: player
- 3: referee

> Run this notebook on Google Colab with a GPU runtime.

## 1. Setup & Installation

In [ ]:
!pip install ultralytics roboflow python-dotenv

In [ ]:
import os
import getpass
from dotenv import load_dotenv
from ultralytics import YOLO
from roboflow import Roboflow

# Load configuration from local .env if present
load_dotenv()

## 2. Load Configuration & Download Dataset

API credentials are read securely without hardcoding:
1. **Google Colab Secrets**: Add `ROBOFLOW_API_KEY` to the 🔑 *Secrets* tab on the left panel.
2. **Local `.env` file**: Picked up automatically via `python-dotenv` if uploaded.
3. **Interactive Prompt**: Prompts for key using masked input (`getpass`) if not found in environment.

In [ ]:
# ── 1. Secure API Key Resolution ──────────────────────────────────
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except (ImportError, Exception):
    ROBOFLOW_API_KEY = None

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = os.getenv('ROBOFLOW_API_KEY')

if not ROBOFLOW_API_KEY:
    ROBOFLOW_API_KEY = getpass.getpass('Enter your Roboflow API Key: ')

# ── 2. Dataset Parameters from Environment ──────────────────────
WORKSPACE_NAME = os.getenv('ROBOFLOW_WORKSPACE', 'shreys-workspace-rkusb')
PROJECT_NAME = os.getenv('ROBOFLOW_PROJECT', 'football-players-detection-3zvbc')
VERSION_NUMBER = int(os.getenv('ROBOFLOW_VERSION', '1'))

# ── 3. Download Dataset ──────────────────────────────────────────
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_NAME).project(PROJECT_NAME)
version = project.version(VERSION_NUMBER)
dataset = version.download('yolov8')

print(f'\n[SUCCESS] Dataset downloaded to: {dataset.location}')

## 3. Verify Dataset Structure

The dataset should have the following YOLO-format structure:
```
dataset/
├── data.yaml
├── train/
│   ├── images/
│   └── labels/
├── valid/
│   ├── images/
│   └── labels/
└── test/
    ├── images/
    └── labels/
```

In [ ]:
# Print data.yaml to verify classes
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, "r") as f:
    print(f.read())

# Count training images
train_images = os.listdir(os.path.join(dataset.location, "train", "images"))
print(f"\nTraining images: {len(train_images)}")

## 4. Fine-Tune YOLOv8x

Training for **100 epochs** on 4 classes:
- 0: ball
- 1: goalkeeper
- 2: player
- 3: referee

In [ ]:
# Load pre-trained YOLOv8x model
model_name = os.getenv("MODEL_NAME", "yolov8x.pt")
epochs = int(os.getenv("EPOCHS", "100"))
batch_size = int(os.getenv("BATCH_SIZE", "16"))
img_size = int(os.getenv("IMAGE_SIZE", "640"))

model = YOLO(model_name)

# Fine-tune on football dataset
results = model.train(
    data=data_yaml_path,
    epochs=epochs,
    imgsz=img_size,
    batch=batch_size,
    device=0,
    project="football_detection",
    name="yolov8x_football",
    patience=20,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

## 5. Evaluate Model

In [ ]:
# Validate on the validation set
metrics = model.val()

print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

# Per-class AP
class_names = ["ball", "goalkeeper", "player", "referee"]
for i, name in enumerate(class_names):
    if i < len(metrics.box.ap50):
        print(f"  {name}: AP50={metrics.box.ap50[i]:.4f}")

## 6. Export Best Weights

Download `best.pt` and place it in `models/best.pt` in the project directory.

In [ ]:
import shutil

# Path to the best weights from training
best_weights_path = os.path.join(
    "football_detection", "yolov8x_football", "weights", "best.pt"
)

# Copy to a convenient location for download
output_path = "/content/best.pt"
shutil.copy2(best_weights_path, output_path)

print(f"Best weights saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1e6:.1f} MB")
print("\nDownload this file and place it at: models/best.pt")

In [ ]:
# For Google Colab: trigger download dialog
try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("Not running on Colab — manually download best.pt from the file browser.")

## 7. Quick Inference Test

In [ ]:
# Run inference on a sample image from the test set
test_images_dir = os.path.join(dataset.location, "test", "images")
test_images = os.listdir(test_images_dir)

if test_images:
    sample_image = os.path.join(test_images_dir, test_images[0])
    results = model.predict(sample_image, conf=0.1, save=True)
    print(f"Inference result saved. Detections: {len(results[0].boxes)}")
    
    # Show the result
    from IPython.display import Image, display
    pred_dir = results[0].save_dir
    pred_image = os.path.join(pred_dir, test_images[0])
    if os.path.exists(pred_image):
        display(Image(filename=pred_image))
else:
    print("No test images found.")